<a href="https://colab.research.google.com/github/rokeya160210/Skin-Cancer-2-papers-comparison/blob/main/05_build_release_package.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Build the release package (#21)

Collects everything the reviewer asked to be released into one folder (and a zip) that can go to GitHub or Zenodo:

```
release/
  README.md                         what is where, how to reproduce every table
  test/derm7pt_test_ids_ordered.csv ordered N-image test cohort (position, case_num, file, diagnosis, label)
  splits/<arm>/seed<S>/{train,val}_ids.csv   image_id, lesion_id, dx, label  (lesion-disjoint)
  predictions/<arm>/seed<S>/prob_<stream>_{test,val}[_bestval].csv  (+ .npy)   N x 6 softmax matrices
  predictions/evit/seed<S>/prob_evit_test[_last].csv
  manifests/  histories/  environment/   run manifests, per-epoch logs, pip freeze per run
  code/                             notebooks 01-05, EViT notebooks, preprocessing.py
  checksums.sha256
```

**No images are redistributed.** HAM10000 and Derm7pt keep their own licences. The package holds identifiers, labels and model outputs only. Model checkpoints are large (ConvNeXt-Large ≈ 750 MB per seed), so they are excluded by default. Set `INCLUDE_CHECKPOINTS = True` only if you will deposit them on Zenodo.

In [1]:
# ---- connect Google Drive FIRST (before any folder is created) ------------------------------
import os, time
def _drive_is_mounted():
    try:
        return any(" /content/drive" in l for l in open("/proc/mounts"))
    except Exception:
        return os.path.ismount("/content/drive")
def mount_google_drive():
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab -> Google Drive not mounted (local paths are used).")
        return False
    if not _drive_is_mounted() and os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        # A plain local folder (e.g. created by an earlier run before Drive was connected) would hide
        # Google Drive. It is NOT your Drive: move it aside (nothing is deleted).
        stray = f"/content/drive_local_copy_{int(time.time())}"
        os.rename("/content/drive", stray)
        print(f"Moved a local folder that was blocking Google Drive to {stray}")
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError("Google Drive could not be connected. Runtime -> Disconnect and delete runtime, run this "
                           "cell alone, choose the account that owns your Drive and tick 'Select all'. "
                           f"(Colab said: {e})")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is not visible at /content/drive/MyDrive.")
    print("Google Drive connected.")
    return True
IN_COLAB = mount_google_drive()
import json, glob, shutil, hashlib, zipfile, datetime
import numpy as np, pandas as pd
SEEDS = [42, 1000, 50000, 2024, 31415]
ENS_BASE = "/content/drive/MyDrive/skin_ablation_v2"
EVIT_BASE = "/content/drive/MyDrive/evit_runs"
COHORT_CSV = "/content/drive/MyDrive/audit/derm7pt_test_cohort_963.csv"      # notebook 01
NOTEBOOK_DIR = "/content/drive/MyDrive/revision_notebooks"                    # where you keep notebooks 01-05 + EViT copies
REL = "/content/drive/MyDrive/release"
INCLUDE_CHECKPOINTS = False
REPO_URL = "<fill in: GitHub / Zenodo DOI>"
CLASSES = ["MEL", "NV", "BCC", "BKL", "DF", "VASC"]
shutil.rmtree(REL, ignore_errors=True)
for d in ["test", "splits", "predictions", "manifests", "histories", "environment", "code"]:
    os.makedirs(os.path.join(REL, d), exist_ok=True)

Mounted at /content/drive
Google Drive connected.


In [2]:
# ---- prerequisites -----------------------------------------------------------------------
missing = [p for p in [COHORT_CSV, os.path.join(ENS_BASE, "runs")] if not os.path.exists(p)]
if missing:
    raise RuntimeError("Run notebooks 01 and 02 first -- not found: " + ", ".join(missing))
# ---- ordered test cohort ---------------------------------------------------------------
cohort = pd.read_csv(COHORT_CSV).sort_values("test_position")
cohort.to_csv(os.path.join(REL, "test", "derm7pt_test_ids_ordered.csv"), index=False)
test_files = list(cohort["derm_file"])
print("test cohort:", len(cohort))

def write_probs(P, dst_stem, row_ids, id_col):
    np.save(dst_stem + ".npy", P.astype(np.float32))
    df = pd.DataFrame(P, columns=[f"p_{c}" for c in CLASSES]); df.insert(0, id_col, row_ids)
    df.to_csv(dst_stem + ".csv", index=False, float_format="%.7f")

# ---- ensemble arms -----------------------------------------------------------------------
for arm in ["controlled", "h2_original"]:
    for s in SEEDS:
        src = os.path.join(ENS_BASE, "runs", arm, f"seed{s}")
        res = os.path.join(src, "results")
        if not os.path.isdir(res):
            print("missing", res); continue
        sp = os.path.join(REL, "splits", arm, f"seed{s}"); os.makedirs(sp, exist_ok=True)
        val_ids = None
        for f in ["train_split.csv", "val_split.csv"]:
            d = pd.read_csv(os.path.join(res, f))
            if "image_id" not in d:
                d["image_id"] = d["path"].map(lambda p: os.path.splitext(os.path.basename(p))[0])
            d = d[[c for c in ["image_id", "lesion_id", "dx", "label"] if c in d]]
            d.to_csv(os.path.join(sp, f.replace("_split", "_ids")), index=False)
            if f.startswith("val"): val_ids = list(d["image_id"])
        pr = os.path.join(REL, "predictions", arm, f"seed{s}"); os.makedirs(pr, exist_ok=True)
        assert (np.load(os.path.join(res, "y_true.npy")) == cohort["label_index"].values).all()
        for f in sorted(glob.glob(os.path.join(res, "prob_*_*.npy"))):
            stem = os.path.splitext(os.path.basename(f))[0]
            P = np.load(f)
            if "_test" in stem:
                assert P.shape == (len(test_files), 6); write_probs(P, os.path.join(pr, stem), test_files, "derm_file")
            elif "_val" in stem:
                assert len(P) == len(val_ids); write_probs(P, os.path.join(pr, stem), val_ids, "image_id")
        shutil.copy(os.path.join(res, "manifest.json"), os.path.join(REL, "manifests", f"{arm}_seed{s}.json"))
        if os.path.isfile(os.path.join(res, "requirements.txt")):
            shutil.copy(os.path.join(res, "requirements.txt"), os.path.join(REL, "environment", f"requirements_{arm}_seed{s}.txt"))
        for h in glob.glob(os.path.join(src, "checkpoints", "*", "history.csv")):
            shutil.copy(h, os.path.join(REL, "histories", f"{arm}_seed{s}_{os.path.basename(os.path.dirname(h))}.csv"))
        if INCLUDE_CHECKPOINTS:
            for k in glob.glob(os.path.join(src, "checkpoints", "*", "*.keras")):
                dst = os.path.join(REL, "checkpoints", arm, f"seed{s}", os.path.basename(os.path.dirname(k)))
                os.makedirs(dst, exist_ok=True); shutil.copy(k, dst)

# ---- EViT --------------------------------------------------------------------------------
for s in SEEDS:
    out = os.path.join(EVIT_BASE, f"seed{s}", "outputs")
    if not os.path.isdir(out):
        print("missing", out); continue
    pr = os.path.join(REL, "predictions", "evit", f"seed{s}"); os.makedirs(pr, exist_ok=True)
    for stem in ["prob_evit_test", "prob_evit_test_last"]:
        f = os.path.join(out, stem + ".npy")
        if os.path.isfile(f): write_probs(np.load(f), os.path.join(pr, stem), test_files, "derm_file")
    for f, dst in [("manifest_revision.json", f"evit_seed{s}.json"), ("train_log.csv", f"evit_seed{s}_train_log.csv"),
                   ("requirements_freeze.txt", f"requirements_evit_seed{s}.txt"), ("environment.json", f"environment_evit_seed{s}.json")]:
        p = os.path.join(out, f)
        if os.path.isfile(p):
            sub = "manifests" if f.endswith(".json") and "manifest" in f else ("histories" if f.endswith(".csv") else "environment")
            shutil.copy(p, os.path.join(REL, sub, dst))
    sp = os.path.join(REL, "splits", "evit", f"seed{s}"); os.makedirs(sp, exist_ok=True)
    for f in ["train_split.csv", "val_split.csv"]:
        p = os.path.join(out, f)
        if os.path.isfile(p):
            d = pd.read_csv(p); d[[c for c in ["image_id", "lesion_id", "label"] if c in d]].to_csv(os.path.join(sp, f.replace("_split", "_ids")), index=False)

test cohort: 963


In [3]:
PREPROC_SRC = r'''
"""preprocessing.py -- the exact preprocessing used in the paper, copied from the training notebooks.

Ensemble arms (InceptionV3, DenseNet121, ConvNeXt-Large; notebook 02):
    decode -> bilinear resize to 224x224 -> float32 on the 0-255 scale. Rescaling to [0,1] happens INSIDE the
    InceptionV3/DenseNet121 models; ConvNeXt-Large receives 0-255 (Keras ConvNeXt normalises internally).
    Training-only augmentation (inactive at inference): horizontal flip; Gaussian noise SD 10/255 on the [0,1] scale.
EViT-Dens169 (EViT notebook): resize to 600x450 (W x H), adaptive average filter (5x5) + Laplacian sharpening,
    then black-hat hair detection (elliptical SE 9, threshold 10) and Telea inpainting (radius 3); the same chain is
    applied to HAM10000 and Derm7pt; then /255.
"""
import numpy as np


def ensemble_decode(path, accurate=True, size=(224, 224)):
    import tensorflow as tf
    raw = tf.io.read_file(path)
    if accurate:   # controlled (shared) harness
        img = tf.cond(tf.image.is_jpeg(raw),
                      lambda: tf.io.decode_jpeg(raw, channels=3, dct_method="INTEGER_ACCURATE"),
                      lambda: tf.io.decode_png(raw, channels=3))
        img.set_shape([None, None, 3]); img = tf.image.resize(img, size, method="bilinear", antialias=False)
    else:          # H2-original harness
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3]); img = tf.image.resize(img, size)
    return tf.cast(img, tf.float32)          # 0-255


def evit_preprocess(path, H=450, W=600, win=5, se=9, thr=10):
    import cv2
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (W, H))
    x = img.astype(np.float32); mu = cv2.blur(x, (win, win)); musq = cv2.blur(x * x, (win, win))
    var = np.maximum(musq - mu * mu, 0.0); k = np.clip((var - float(var.mean())) / (var + 1e-6), 0.0, 1.0)
    img = np.clip(mu + k * (x - mu) + cv2.Laplacian(x, cv2.CV_32F, ksize=3), 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    bh = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (se, se)))
    _, mask = cv2.threshold(bh, thr, 255, cv2.THRESH_BINARY)
    return cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)   # uint8 RGB; the model input is this / 255
'''

open(os.path.join(REL, "code", "preprocessing.py"), "w").write(PREPROC_SRC)
for f in sorted(glob.glob(os.path.join(NOTEBOOK_DIR, "*.ipynb"))):
    shutil.copy(f, os.path.join(REL, "code"))
print("code/:", sorted(os.listdir(os.path.join(REL, "code"))))

code/: ['preprocessing.py']


In [4]:
readme = f"""# Release package: Cross-dataset external validation of deep ensembles (HAM10000 -> Derm7pt)

Created {datetime.date.today().isoformat()}. Repository / DOI: {REPO_URL}

## Contents
* `test/derm7pt_test_ids_ordered.csv`: the {len(cohort)}-image external cohort **in the exact row order of every prediction matrix**
  (test_position, Derm7pt case_num, dermoscopic file, raw diagnosis, mapped class, label index).
* `splits/<arm>/seed<S>/`: lesion-disjoint HAM10000 train/validation identifiers (image_id, lesion_id, dx, label).
* `predictions/<arm>/seed<S>/prob_<stream>_test.csv`: N x 6 softmax outputs (columns p_MEL..p_VASC), one row per test image.
  `_val` = HAM10000 validation outputs (keyed by image_id); `_bestval` = min-validation-loss checkpoint (sensitivity analysis).
  The fused configurations are reproduced as weighted sums of these matrices (weights in the paper and in notebook 03).
* `manifests/`: one JSON per run (seed, harness, split fingerprints, augmentation, checkpoint policy, software versions).
* `histories/`: per-epoch training/validation loss and accuracy.
* `environment/`: `pip freeze` of every run (TensorFlow 2.20.0, Keras 3.13.2).
* `code/`: notebooks 01 (cohort + overlap audit), 02 (training), 03 (all tables/figures), 04 (profiling), 05 (this package),
  the EViT-Dens169 notebooks, and `preprocessing.py`.

## Reproducing the tables without retraining
Run notebook 03 with `RUNS`/`EVIT_RUNS` pointed at `predictions/`. Every number in the paper is computed from these matrices.

## Data licences
No images are included. Obtain HAM10000 (Tschandl et al., 2018) and Derm7pt (Kawahara et al., 2019) from their official sources
under their respective terms; the identifiers here map one-to-one onto those releases.

## Seeds
{SEEDS}, fixed before the revision runs.
"""
open(os.path.join(REL, "README.md"), "w").write(readme)

# checksums + zip
lines = []
for root, _, files in os.walk(REL):
    for f in sorted(files):
        p = os.path.join(root, f)
        if f == "checksums.sha256": continue
        lines.append(f"{hashlib.sha256(open(p,'rb').read()).hexdigest()}  {os.path.relpath(p, REL)}")
open(os.path.join(REL, "checksums.sha256"), "w").write("\n".join(sorted(lines, key=lambda l: l.split('  ')[1])) + "\n")
zp = REL.rstrip("/") + ".zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(REL):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), os.path.dirname(REL)))
print(f"{len(lines)} files; zip: {zp} ({os.path.getsize(zp)/2**20:.1f} MB)")

318 files; zip: /content/drive/MyDrive/release.zip (5.6 MB)
